# Federated learning with PyTorch modules (non-IID)

Same federated setup as `fed_learning_cnn_non_iid.ipynb`: same seeds, same Dirichlet split (every client gets the identical shard), client selection, size-weighted FedAvg and LR decay. The difference is the model: an `nn.Module` trained with autograd and `torch.optim.Adam`, and the server averages `state_dict()`. The hand-written notebooks remain the from-scratch reference.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets
import random
import copy

# For reproducibility
np.random.seed(42)
random.seed(42)        # client selection
torch.manual_seed(42)  # weight init, dropout, random_split

# Device configuration (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

learning_rate = 0.005
batch_size = 64

epochs = 3             # local epochs per round (fewer local epochs + more rounds reduces client drift)
num_clients = 10       # number of clients in federated learning
federated_rounds = 10  # number of rounds of federated training
selection_factor = 0.8 # fraction of clients selected each round

lr_decay_factor = 0.5  # multiply the clients' learning rate by this every lr_decay_every rounds
lr_decay_every = 3     # how often (in rounds) to decay; scale with federated_rounds or lr decays to ~0 early

dirichlet_alpha = 0.5  # label skew across clients: smaller = more non-IID (0.1 extreme, 100 ~ IID)

model_type = "cnn"     # "cnn" or "mlp"

In [ ]:
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True)

# Whole dataset as tensors once, scaled like transforms.ToTensor(): (N, 1, 28, 28) in [0, 1]
X_train_all = train_dataset.data.float().div(255).unsqueeze(1)
y_train_all = train_dataset.targets
X_test = test_dataset.data.float().div(255).unsqueeze(1)
y_test = test_dataset.targets

In [ ]:
# Non-IID split: for each class, divide its samples across clients using proportions
# drawn from Dirichlet(alpha). Small alpha -> each client sees only a few classes.
# Returns, per client, indices into the full training set.
def dirichlet_split(subset, num_clients, alpha, min_size=10):
    labels = torch.as_tensor(subset.dataset.targets).numpy()[subset.indices]
    num_classes = labels.max() + 1

    # Resample until every client has at least min_size samples (tiny alpha can leave a client empty)
    while True:
        client_indices = [[] for _ in range(num_clients)]
        for c in range(num_classes):
            idx_c = np.where(labels == c)[0]
            np.random.shuffle(idx_c)
            proportions = np.random.dirichlet([alpha] * num_clients)
            cuts = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            for client_id, part in enumerate(np.split(idx_c, cuts)):
                client_indices[client_id].extend(part.tolist())
        if min(len(ci) for ci in client_indices) >= min_size:
            break

    subset_indices = np.array(subset.indices)
    client_datasets = []
    for client_id, ci in enumerate(client_indices):
        # Indices were appended class by class; training batches in order, so unshuffled
        # data would give single-class batches
        np.random.shuffle(ci)
        counts = np.bincount(labels[ci], minlength=num_classes)
        print(f"Client {client_id}: {len(ci)} samples, per-class {counts.tolist()}")
        client_datasets.append(torch.as_tensor(subset_indices[ci]))

    return client_datasets

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),    # 32 x 28 x 28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # 32 x 14 x 14
            nn.Conv2d(32, 64, 3, padding=1),   # 64 x 14 x 14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # 64 x 7 x 7
            nn.Flatten(),                      # 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.layers(x)  # logits; CrossEntropyLoss applies the softmax


# Same 784-128-64-10 MLP as the hand-written notebooks
class MLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.layers(x)


def build_model(model_type):
    if model_type == "cnn":
        return CNN()
    if model_type == "mlp":
        return MLP()
    raise ValueError(f"model_type must be 'cnn' or 'mlp', got {model_type!r}")

In [ ]:
# Federated Learning Client Class
class Client:
    def __init__(self, model, X_train, y_train, epochs, batch_size, device='cpu'):
        self.X_train = X_train
        self.y_train = y_train
        self.num_samples = len(y_train)
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = device
        # Private copy: sharing one model object makes every client train the same tensors
        self.model = copy.deepcopy(model)

    def train(self, lr):
        # Fresh optimizer every round, like the hand-written version: Adam state starts at zero
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        self.model.train()

        for epoch in range(self.epochs):
            total_loss, correct = 0.0, 0
            for i in range(0, self.num_samples, self.batch_size):
                x_batch = self.X_train[i:i+self.batch_size].to(self.device)
                y_batch = self.y_train[i:i+self.batch_size].to(self.device)

                optimizer.zero_grad()
                logits = self.model(x_batch)
                loss = loss_fn(logits, y_batch)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * len(y_batch)
                correct += (logits.argmax(dim=1) == y_batch).sum().item()

            # Accuracy over this epoch's training batches (train mode), so no extra pass over the data
            print(f"Epoch {epoch+1}/{self.epochs}, Train Loss: {total_loss / self.num_samples:.4f}, "
                  f"Train Acc: {correct / self.num_samples:.4f}")

    def get_parameters(self):
        # clone(): the optimizer keeps updating the live tensors
        return {key: value.detach().clone() for key, value in self.model.state_dict().items()}

    def receive_model(self, global_model):
        self.model = copy.deepcopy(global_model)

In [ ]:
class Server:

    def __init__(self, model, X_val, y_val, X_test, y_test, learning_rate=0.005, device="cpu"):
        self.global_model = model
        self.clientList = [] #List of all clients registered
        self.selectedClientList = [] #List of clients selected in a specific round
        self.learning_rate = learning_rate  # the clients' Adam lr, decayed across rounds
        self.device = device
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test

    # Register Clients

    def register(self, client):
        self.clientList.append(client)
        print("Registering client")

    # Select Clients in each round

    def select(self, clientMap):
        client_id, client = clientMap
        self.selectedClientList.append(clientMap)
        print("Selecting client ", client_id)

    # Reset the server in each round

    def reset(self):
        self.selectedClientList = []

    # Aggregrate updates

    def aggregate(self, localUpdates, client_sizes):

        print("Getting aggregated by server")

        total_samples = sum(client_sizes)
        aggregated_params = {}
        for key, value in localUpdates[0].items():
            if not value.is_floating_point():
                # BatchNorm's num_batches_tracked counter: an integer, not a weight to average
                aggregated_params[key] = value.clone()
                continue
            # FedAvg weighted by each client's local sample count (n_k/n), not a plain mean
            aggregated_params[key] = sum(update[key] * (size / total_samples)
                                         for update, size in zip(localUpdates, client_sizes))

        return aggregated_params

    # Update global model with aggregrated parameters

    def update_global_model(self, localUpdates, client_sizes):
        self.global_model.load_state_dict(self.aggregate(localUpdates, client_sizes))
        print("Done Aggregating")

    def decay_learning_rate(self, factor):
        self.learning_rate *= factor
        print(f"Decayed learning rate by {factor}, lr now {self.learning_rate:.6f}")

    def accuracy(self, X, y, chunk_size=1000):
        self.global_model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, len(y), chunk_size):
                logits = self.global_model(X[i:i+chunk_size].to(self.device))
                correct += (logits.argmax(dim=1).cpu() == y[i:i+chunk_size]).sum().item()
        return correct / len(y)

    def evaluate(self):
        val_accuracy = self.accuracy(self.X_val, self.y_val)
        test_accuracy = self.accuracy(self.X_test, self.y_test)
        print(f"Val Acc: {val_accuracy:.4f}, Test Acc: {test_accuracy:.4f}")

    # Distribute updated model to clients

    def distribute_global_model(self):
        for client in self.clientList:
            client.receive_model(self.global_model)

In [ ]:
# Main Federated Learning Process
def federated_learning_simulation(epochs, batch_size, learning_rate, num_clients, federated_rounds, selection_factor, lr_decay_factor, lr_decay_every, dirichlet_alpha, model_type):

    # Hold the validation split out first, so no client ever trains on it
    val_size = len(train_dataset) // 5
    train_subset, val_subset = torch.utils.data.random_split(train_dataset, [len(train_dataset) - val_size, val_size])
    val_indices = torch.as_tensor(val_subset.indices)

    # Validation/test stay IID (global class balance); only the client training shards are skewed
    client_indices = dirichlet_split(train_subset, num_clients, dirichlet_alpha)

    global_model = build_model(model_type).to(device)

    server = Server(global_model, X_train_all[val_indices], y_train_all[val_indices], X_test, y_test,
                    learning_rate=learning_rate, device=device)

    for i in range(num_clients):
        indices = client_indices[i]
        client = Client(global_model, X_train_all[indices], y_train_all[indices], epochs, batch_size, device)
        server.register(client)

    for round in range(federated_rounds):

        server.reset()

        for i in range(num_clients):
            rd = random.randint(0, 100)
            if rd < selection_factor * 100:
                server.select((i, server.clientList[i]))

        client_weights = []
        selected_client_sizes = []

        for client_id, client in server.selectedClientList:
            print("Training started by client ", client_id)
            client.train(server.learning_rate)
            client_weights.append(client.get_parameters())
            selected_client_sizes.append(client.num_samples)
            print("Training completed by client ", client_id)

        server.update_global_model(client_weights, selected_client_sizes)
        server.evaluate()

        if (round + 1) % lr_decay_every == 0:
            server.decay_learning_rate(lr_decay_factor)

        server.distribute_global_model()


federated_learning_simulation(epochs, batch_size, learning_rate, num_clients, federated_rounds, selection_factor, lr_decay_factor, lr_decay_every, dirichlet_alpha, model_type)